In [1]:
import pandas as pd 
import os
import re
import itertools
import random
import numpy as np

## Text manipulation

### Text extraction

In [2]:
def get_raw_data(folder):
    file_paths = []

    for root, dirs, files in os.walk(folder):
        for name in files:
            if name.endswith(".txt") and name.startswith('student'):
                file_paths.append(os.path.join(root, name))

    all_txts = []
    for file_path in file_paths:
        txt = open(file_path).read()
        path = re.search(r'test_studenti_gpt_4o/(.*)', file_path)
        trimmed_path = path.group(1) if path else file_path
        all_txts.append({'txt' : txt, 
                         'file_path' : trimmed_path
                         })

    all_txts = pd.DataFrame(all_txts)
    print(f"Number of generated files: {len(file_paths)}")
    return all_txts

### Text cleaning

In [3]:
def process_txt_dataset(all_txts, dataset_name):
    data = []

    for i, txt in enumerate(all_txts['txt']):
        # create participant_id
        participant_id = f"{dataset_name}_{i+1:03d}"

        # base text
        mod_text = txt

        # --- CLEANING ---
        # case: 4NaNNaN -> remove NaN repetition
        value_nan = re.search(r'(\d)(NaN){1,3}', mod_text)
        if value_nan:
            mod_text = mod_text.replace(value_nan.group(2), '')

        # remove standalone NaN
        mod_text = re.sub(r'\bNaN\b', '', mod_text)

        # case: determinazionenan5 -> determinazione,5
        mod_text = re.sub(r'\b(\w*?)nan(\d)\b', r'\1,\2', mod_text, flags=re.IGNORECASE)

        # remove junk text before first "fisica"
        fisica_text = re.search(r'(?im)(^.*?)(fisica,\d+)', mod_text)
        if fisica_text:
            mod_text = mod_text.replace(fisica_text.group(1), '')

        # --- PROMPT EXTRACTION ---
        prompt = None
        prompt_match = re.search(r'(?s)(.*?)(\n[Ff]isica,\d+)', mod_text)
        if prompt_match:
            prompt = prompt_match.group(1)
            # drop prompt text but keep "fisica"
            mod_text = mod_text.replace(str(prompt_match.group(1)), '')

        # --- MAS-IT EXTRACTION (23 numbers comma-separated) ---
        mas_it = None
        lines = mod_text.splitlines()
        for line in lines:
            line = line.strip()
            match = re.search(r'^\s*(?:[1-5],){22}[1-5]\s*$', line)
            if match:
                mas_it = match.group()
                # remove mas_it line from text
                mod_text = mod_text.replace(match.group(), '')

        # --- ASSOCIATIONS EXTRACTION ---
        filtered_lines = [
            line for line in lines
            if re.match(r"^([a-zA-Z0-9_\-']+,\d+,){2,}[a-zA-Z0-9_\-']+,\d+(,*)$", line)
        ]
        mod_text = '\n'.join(filtered_lines)

        # --- COLLECT EVERYTHING ---
        data.append({
            "participant_id": participant_id,
            "file_path": all_txts["file_path"].iloc[i] if "file_path" in all_txts else None,
            "prompt": prompt,
            "mas_it": mas_it,
            "associations": mod_text,
        })

    return pd.DataFrame(data)

## Personal information extraction

In [4]:
def extract_personal_info(df):
    
    ages, genders, enrolment_years, faculties, conditions = [], [], [], [], []

    for prompt in df['prompt'].fillna(""):
        # --- AGE ---
        age_match = re.search(r'\b(\d{1,3})\s+anni\b', prompt)
        ages.append(age_match.group(1) if age_match else None)

        # --- GENDER ---
        if re.search(r'\bSei una studentessa\b', prompt):
            genders.append("F")
        elif re.search(r'\bSei uno studente\b', prompt):
            genders.append("M")
        else:
            genders.append(None)

        # --- ENROLMENT YEAR ---
        enrolment_match = re.search(r'\b(\d+)°\s+anno', prompt)
        enrolment_years.append(enrolment_match.group(1) if enrolment_match else None)

        # --- FACULTY / SCHOOL ---
        uni_match = re.search(r'universita alla facolta di\s+(\w+)', prompt, flags=re.IGNORECASE)
        liceo_match = re.search(r'scuola superiore al liceo\s+(\w+)', prompt, flags=re.IGNORECASE)
        if uni_match:
            faculties.append(f"uni_{uni_match.group(1)}")
        elif liceo_match:
            faculties.append(f"liceo_{liceo_match.group(1)}")
        else:
            faculties.append(None)

        # --- SOCIO-ECONOMIC CONDITION ---
        socio_match = re.search(r'vivi in condizioni socio-economiche\s+([\w\-]+)', prompt, flags=re.IGNORECASE)
        conditions.append(socio_match.group(1) if socio_match else None)

    # --- Add to DataFrame ---
    df = df[['participant_id', 'file_path', 'mas_it']].copy()  # Keep only participant_id to avoid modifying original df
    df["age"] = ages
    df["gender"] = genders
    df["year"] = enrolment_years
    df["faculty"] = faculties
    df["condition"] = conditions

    return df

## MAS-IT

In [5]:
def all_questionnaires(df):
    
    # Split MAS-IT responses
    df_mas_it = df['mas_it'].str.split(",", expand=True)
    
    # Label the columns
    df_mas_it.columns = [f"mas_it_{i+1}" for i in range(23)]
    
    df_mas_it.insert(0, 'participant_id', df['participant_id'])

    # mapped scores with italian order to the origianl order
    correspondence_order = [
        'participant_id', 'mas_it_1', 'mas_it_3', 'mas_it_6', 'mas_it_7', 'mas_it_10', 'mas_it_18', 'mas_it_19',
        'mas_it_21', 'mas_it_23', 'mas_it_2', 'mas_it_4', 'mas_it_5', 'mas_it_8', 'mas_it_11',
        'mas_it_13', 'mas_it_14', 'mas_it_22', 'mas_it_9', 'mas_it_12', 'mas_it_15',
        'mas_it_16', 'mas_it_17', 'mas_it_20'
    ]

    # reorder columns based on the original order
    mas_it_reordered = df_mas_it[correspondence_order]

    # # name columns to match the original names
    mas_it_reordered.columns = df_mas_it.columns

    id_cols=["participant_id", "age", "gender", "faculty", "year", "condition"]
    
    # Concatenate with id/personal info
    df_questionnaire = df[id_cols]

    return df_questionnaire, mas_it_reordered

## Associations-Valence task

In [6]:
def process_associations(df):

    df_assoc = df[["participant_id", "associations"]].copy()

    # Split by newline and explode
    df_assoc["associations"] = df_assoc["associations"].str.split("\n")
    df_assoc = df_assoc.explode("associations").reset_index(drop=True)

    # Remove empty lines
    df_assoc = df_assoc[df_assoc["associations"].str.strip().astype(bool)]

    # Split into separate columns
    col_names = [
        "cue_word", "valence_cue_word",
        "association_1", "valence_association_1",
        "association_2", "valence_association_2",
        "association_3", "valence_association_3"
    ]
    df_assoc[col_names] = df_assoc["associations"].str.split(",", expand=True)

    # Drop original column
    df_assoc = df_assoc.drop(columns=["associations"])

    # Lowercase certain columns
    for col in ["cue_word", "association_1", "association_2", "association_3"]:
        df_assoc[col] = df_assoc[col].str.lower()

    # Correct known typos
    df_assoc.loc[df_assoc["cue_word"] == "psicoogia", "cue_word"] = "psicologia"

    # Sort
    df_assoc = df_assoc.sort_values(by=[ "cue_word", "participant_id"]).reset_index(drop=True)

    return df_assoc


## Check for missing data

In [11]:
def find_missing_cue_words(df_associations, df_personal_data):
    # Get unique cue words and participants
    cue_words = df_associations['cue_word'].unique().tolist()
    participants = df_associations['participant_id'].unique().tolist()

    print(f"Number of cue words: {len(cue_words)}")
    print(f"Number of participants: {len(participants)}")

    # Build all possible participant × cue_word pairs
    all_combinations = pd.DataFrame(
        itertools.product(participants, cue_words),
        columns=['participant_id', 'cue_word']
    )

    # Compare against actual associations
    merged = all_combinations.merge(
        df_associations[['participant_id', 'cue_word']],
        how="left", indicator=True
    )

    # Keep only missing ones
    missing_pairs = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])

    # Add file_path info from metadata
    missing_info = df_personal_data[['participant_id', 'file_path']]
    missing_df = missing_pairs.merge(missing_info, on='participant_id', how='left')

    # Output
    if missing_pairs.empty:
        print("No missing cue words ✅")
    else:
        print("⚠️ Missing cue words found")
        print(missing_pairs)
    
    len_cue_words = []
    # get how many cue words are in the df, to see wich one are missing
    for cue_word in cue_words:
        len_word = df_associations.loc[df_associations['cue_word'] == cue_word, 'cue_word']
        len_cue_words.append(len(len_word))
        # if there arent 356 cue words (number of participants)
        if len(len_word) != len(df_personal_data):
            print(f'{cue_word} : {len(len_word)}')
    else:
        print("All the cue words are present for each participant ✅")


# Pipeline application on data

## GPT-3.5

In [8]:
abs_path = '/Users/emma/projects/cognoscolab/math_anxiety_LLMs'

In [12]:
folder_gpt_3_5 = f'{abs_path}/data/gpt_3_5_students_data'

raw_data_gpt_3_5 = get_raw_data(folder_gpt_3_5)
cleaned_data_gpt_3_5 = process_txt_dataset(raw_data_gpt_3_5, 'gpt_3_5')
personal_data_gpt_3_5 = extract_personal_info(cleaned_data_gpt_3_5)
all_questionnaires_gpt_3_5, mas_it_gpt_3_5 = all_questionnaires(personal_data_gpt_3_5)
association_valence_gpt_3_5 = process_associations(cleaned_data_gpt_3_5)
find_missing_cue_words(association_valence_gpt_3_5, personal_data_gpt_3_5)

Number of generated files: 300
Number of cue words: 40
Number of participants: 300
No missing cue words ✅
All the cue words are present for each participant ✅


## GPT-4o

In [13]:
folder_gpt_4o = f'{abs_path}/data/gpt_4o_students_data'

raw_data_gpt_4o = get_raw_data(folder_gpt_4o)
cleaned_data_gpt_4o = process_txt_dataset(raw_data_gpt_4o, 'gpt_4o')
personal_data_gpt_4o = extract_personal_info(cleaned_data_gpt_4o)
all_questionnaires_gpt_4o, mas_it_gpt_4o = all_questionnaires(personal_data_gpt_4o)
association_valence_gpt_4o = process_associations(cleaned_data_gpt_4o)
find_missing_cue_words(association_valence_gpt_4o, personal_data_gpt_4o)

Number of generated files: 356
Number of cue words: 40
Number of participants: 356
No missing cue words ✅
All the cue words are present for each participant ✅


## Remove participats to have 300 students for GPT-4o

In [32]:
# all the rows with empty values in the association task
empty_row_associations = association_valence_gpt_4o[association_valence_gpt_4o.isnull().any(axis=1)]
print(f"Participant with some empty rows in the associations task: {empty_row_associations['participant_id'].unique()}")
# all the rows with empty values in the mas-it task
empty_rows_mas_it = mas_it_gpt_4o[mas_it_gpt_4o.isnull().any(axis=1)]
print(f"Participant with some empty rows in the mas-it: {empty_rows_mas_it['participant_id'].unique()}")

# combine participants with empty rows in either associations OR mas_it
to_remove = np.union1d(
    empty_row_associations['participant_id'].unique(),
    empty_rows_mas_it['participant_id'].unique()
)
# remove from the associations df the participants with empty rows in the associations task and in the mas-it task
association_valence_gpt_4o_cleaned = association_valence_gpt_4o[~association_valence_gpt_4o['participant_id'].isin(to_remove)]

mas_it_gpt_4o_cleaned = mas_it_gpt_4o[~mas_it_gpt_4o['participant_id'].isin(to_remove)]

# creates a list of participant_ids left after removing the ones with empty rows
participant_ids_left = mas_it_gpt_4o_cleaned['participant_id']

random.seed(42)  # for reproducibility
# randomly select 44 participant_ids to remove from the ones left -> to get 300 students
participant_ids_to_remove = random.sample(list(participant_ids_left), len(participant_ids_left) - 300)
print(f"Randomly selected participant_ids to remove: {participant_ids_to_remove}")

# remove the selected participant_ids from the associations and mas-it df
association_valence_gpt_4o_cleaned = association_valence_gpt_4o_cleaned[~association_valence_gpt_4o_cleaned['participant_id'].isin(participant_ids_to_remove)]

mas_it_gpt_4o_cleaned = mas_it_gpt_4o_cleaned[~mas_it_gpt_4o_cleaned['participant_id'].isin(participant_ids_to_remove)]

all_questionnaires_gpt_4o_cleaned = all_questionnaires_gpt_4o[~all_questionnaires_gpt_4o['participant_id'].isin(np.concatenate([participant_ids_to_remove, to_remove]))]

Participant with some empty rows in the associations task: ['gpt_4o_301' 'gpt_4o_073' 'gpt_4o_154' 'gpt_4o_115' 'gpt_4o_271'
 'gpt_4o_066']
Participant with some empty rows in the mas-it: ['gpt_4o_059' 'gpt_4o_082' 'gpt_4o_089' 'gpt_4o_134' 'gpt_4o_168'
 'gpt_4o_184']
Randomly selected participant_ids to remove: ['gpt_4o_340', 'gpt_4o_058', 'gpt_4o_013', 'gpt_4o_148', 'gpt_4o_132', 'gpt_4o_121', 'gpt_4o_075', 'gpt_4o_053', 'gpt_4o_291', 'gpt_4o_045', 'gpt_4o_315', 'gpt_4o_227', 'gpt_4o_017', 'gpt_4o_016', 'gpt_4o_048', 'gpt_4o_118', 'gpt_4o_126', 'gpt_4o_269', 'gpt_4o_321', 'gpt_4o_014', 'gpt_4o_299', 'gpt_4o_107', 'gpt_4o_345', 'gpt_4o_225', 'gpt_4o_119', 'gpt_4o_240', 'gpt_4o_314', 'gpt_4o_150', 'gpt_4o_004', 'gpt_4o_086', 'gpt_4o_185', 'gpt_4o_084', 'gpt_4o_117', 'gpt_4o_182', 'gpt_4o_205', 'gpt_4o_050', 'gpt_4o_194', 'gpt_4o_187', 'gpt_4o_322', 'gpt_4o_143', 'gpt_4o_023', 'gpt_4o_246', 'gpt_4o_286', 'gpt_4o_065']


# Exporting to csv

In [33]:
association_valence_gpt_3_5.to_csv(f'{abs_path}/out/formatting/gpt_3_5/associations_valence_gpt_3_5.csv', index=False)

all_questionnaires_gpt_3_5.to_csv(f'{abs_path}/out/formatting/gpt_3_5/all_questionnaires_gpt_3_5.csv', index=False)

mas_it_gpt_3_5.to_csv(f'{abs_path}/out/formatting/gpt_3_5/mas_it_gpt_3_5.csv', index=False)

In [34]:
association_valence_gpt_4o_cleaned.to_csv(f'{abs_path}/out/formatting/gpt_4o/associations_valence_gpt_4o.csv', index= False)

all_questionnaires_gpt_4o_cleaned.to_csv(f'{abs_path}/out/formatting/gpt_4o/all_questionnaires_gpt_4o.csv', index= False)

mas_it_gpt_4o_cleaned.to_csv(f'{abs_path}/out/formatting/gpt_4o/mas_it_gpt_4o.csv', index= False)